# 09 · Guest Experience Intelligence — interactive dashboard
Everything from notebooks 01–08 in one place: sentiment, topics, topic-level sentiment, hotel / location / segment
differences, model quality and a review explorer.

**How to use**
1. **Part A** shows a static snapshot of every view with no filters (readable anywhere).
2. **Part B** (last section) is a live dashboard: pick filters, press **Apply**, switch tabs.
   It needs a running kernel (JupyterLab / VS Code / Colab) with `ipywidgets` and `plotly`.

Requires `outputs/dashboard/*.csv` from notebook 08.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
import ipywidgets as widgets
from IPython.display import HTML, display
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"; DASH = OUT / "dashboard"; TOP = OUT / "topics"
LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#D1495B", "neutral": "#B8BCC4", "positive": "#2E8B57"}
GENERAL = "General / Overall"
SEG_DIMS = {"Customer type": "customer_type", "Loyalty status": "loyalty_status", "Booking channel": "booking_channel",
            "Room type": "room_type", "Guest country": "country", "Hotel star rating": "star_rating",
            "Length of stay": "stay_bucket", "Previous stays": "prev_bucket"}
FILTERS = [("hotel_name", "Hotel"), ("location", "City"), ("country", "Guest country"), ("customer_type", "Customer type"),
           ("loyalty_status", "Loyalty"), ("booking_channel", "Channel"), ("room_type", "Room type"), ("star_rating", "Stars")]
DIMS = ["review_id", "hotel_name", "location", "country", "customer_type", "loyalty_status", "booking_channel",
        "room_type", "star_rating", "stay_bucket", "prev_bucket", "month"]

rev_all = pd.read_csv(DASH / "reviews_enriched.csv", parse_dates=["review_date"])
rev_all["loyalty_status"] = rev_all["loyalty_status"].fillna("None")
rev_all["month"] = rev_all.review_date.dt.to_period("M").dt.to_timestamp()
rev_all["stay_bucket"] = pd.cut(rev_all.stay_duration, [0, 1, 3, 5, 100], labels=["1 night", "2-3", "4-5", "6+"])
rev_all["prev_bucket"] = pd.cut(rev_all.previous_stays, [-1, 0, 2, 5, 1000], labels=["0", "1-2", "3-5", "6+"])
rev_all["topics"] = rev_all.topics.fillna("")
sen_all = pd.read_csv(DASH / "sentences_enriched.csv")
print(f"{len(rev_all):,} reviews · {len(sen_all):,} sentences")

34,460 reviews · 101,077 sentences


## Helpers: filtering, aggregation, chart builders

In [2]:
def apply_filters(f):
    m = rev_all.review_date.dt.date.between(f["d0"], f["d1"])
    for col, _ in FILTERS:
        if f.get(col): m &= rev_all[col].isin(f[col])
    if f.get("drop_issues"): m &= ~rev_all.label_issue
    rev = rev_all[m].copy()
    rev["sent"] = rev["pred_label"] if f["source"] == "Model-predicted" else rev["sentiment_label"]
    sen = sen_all[sen_all.review_id.isin(rev.review_id)].merge(rev[DIMS], on="review_id")
    return rev, sen[sen.category != GENERAL]

def summarise(df, col, by):
    g = df.groupby([by, col], observed=True).size().unstack(fill_value=0).reindex(columns=LABELS, fill_value=0)
    n = g.sum(axis=1); out = g.div(n, axis=0) * 100; out["n"] = n; out["net"] = out.positive - out.negative
    return out.reset_index()

def _layout(fig, title, h):
    fig.update_layout(title=title, height=h, margin=dict(l=0, r=10, t=45, b=0), legend=dict(orientation="h", y=-0.15))
    return fig

def stacked(df, by, title, h=380):
    fig = go.Figure()
    for l in LABELS:
        fig.add_bar(y=df[by].astype(str), x=df[l], name=l, orientation="h", marker_color=COLORS[l], customdata=df.n,
                    hovertemplate="%{y}<br>" + l + ": %{x:.1f}%<br>n=%{customdata:,}<extra></extra>")
    fig.update_layout(barmode="stack", xaxis_title="%"); return _layout(fig, title, h)

def net_bar(df, by, title, h=400):
    d = df.sort_values("net")
    fig = go.Figure(go.Bar(y=d[by].astype(str), x=d.net, orientation="h", customdata=d.n,
                           marker_color=np.where(d.net >= 0, COLORS["positive"], COLORS["negative"]),
                           hovertemplate="%{y}<br>net %{x:.1f}<br>n=%{customdata:,}<extra></extra>"))
    fig.update_layout(xaxis_title="net sentiment (% pos − % neg)"); return _layout(fig, title, h)

def heat(z, title, h=420):
    fig = go.Figure(go.Heatmap(z=z.values, x=[str(c) for c in z.columns], y=[str(i) for i in z.index],
                               colorscale=[[0, COLORS["negative"]], [.5, "#F5F5F5"], [1, COLORS["positive"]]], zmid=0,
                               colorbar=dict(title="net %"), hoverongaps=False))
    return _layout(fig, title, h)

def net_matrix(sen, rows, min_n):
    z = sen.groupby([rows, "category", "sent_pred"], observed=True).size().unstack(fill_value=0).reindex(columns=LABELS, fill_value=0)
    n = z.sum(axis=1)
    return ((z.positive - z.negative) / n * 100).where(n >= min_n).unstack("category")

def kpis(rev):
    p = rev.sent.value_counts(normalize=True).reindex(LABELS).fillna(0) * 100
    items = [("Reviews", f"{len(rev):,}"), ("Avg rating", f"{rev.rating_score.mean():.2f} / 5"), ("Positive", f"{p.positive:.1f}%"),
             ("Neutral", f"{p.neutral:.1f}%"), ("Negative", f"{p.negative:.1f}%"), ("Net sentiment", f"{p.positive - p.negative:+.1f}")]
    box = "<div style='padding:10px 16px;border-radius:8px;background:#f3f4f6;min-width:110px'><div style='font-size:12px;color:#555'>{}</div><div style='font-size:22px;font-weight:600;color:#111'>{}</div></div>"
    return HTML("<div style='display:flex;gap:12px;flex-wrap:wrap;margin:6px 0'>" + "".join(box.format(k, v) for k, v in items) + "</div>")

## Views (each takes already-filtered data)

In [3]:
def view_overview(rev, sen):
    display(kpis(rev))
    m = rev.groupby("month").size().reset_index(name="reviews"); ms = summarise(rev, "sent", "month")
    f1 = go.Figure(go.Bar(x=m.month, y=m.reviews, marker_color="#4C78A8")); _layout(f1, "Review volume by month", 320)
    f2 = go.Figure()
    for l in ("positive", "negative"): f2.add_scatter(x=ms.month, y=ms[l], mode="lines+markers", name=l, line_color=COLORS[l])
    f2.update_layout(yaxis_title="% of reviews"); _layout(f2, "% positive / % negative by month", 320)
    bins = np.linspace(1, 5, 41); f3 = go.Figure()
    for l in LABELS:
        h, _ = np.histogram(rev[rev.sent == l].rating_score, bins=bins); f3.add_bar(x=(bins[:-1] + bins[1:]) / 2, y=h, name=l, marker_color=COLORS[l])
    f3.update_layout(barmode="stack", xaxis_title="rating_score"); _layout(f3, "Rating by sentiment", 320)
    ct = pd.crosstab(rev.sentiment_label, rev.pred_label).reindex(index=LABELS, columns=LABELS, fill_value=0)
    f4 = go.Figure(go.Heatmap(z=ct.values, x=LABELS, y=LABELS, text=ct.values, texttemplate="%{text}", colorscale="Blues", showscale=False))
    f4.update_layout(xaxis_title="model-predicted", yaxis_title="given label"); _layout(f4, "Given label vs model prediction", 320)
    for f in (f1, f2, f3, f4): display(f)
    print(f"Model agrees with the given label on {100 * (rev.sentiment_label == rev.pred_label).mean():.1f}% of the reviews in view.")

def view_topics(rev, sen, basis="Sentence-level"):
    if rev.empty or sen.empty: print("no data"); return
    if basis.startswith("Sentence"):
        ts = summarise(sen, "sent_pred", "category"); ts["reviews"] = ts.category.map(sen.groupby("category").review_id.nunique())
    else:
        rt = rev.assign(category=rev.topics.str.split("; ")).explode("category"); rt = rt[rt.category.fillna("") != ""]
        ts = summarise(rt, "sent", "category"); ts["reviews"] = ts.n
    ts = ts.sort_values("net")
    display(stacked(ts, "category", "Sentiment mix by topic")); display(net_bar(ts, "category", "Net sentiment by topic"))
    tb = ts.rename(columns={"n": "mentions", "negative": "% neg", "neutral": "% neu", "positive": "% pos"}).round(1)
    tb["note"] = np.where(tb.mentions < 300, "⚠ low volume", ""); display(tb[["category", "reviews", "mentions", "% neg", "% neu", "% pos", "net", "note"]])
    # drivers
    rt = rev.assign(category=rev.topics.str.split("; ")).explode("category"); rt = rt[rt.category.fillna("") != ""]
    cls = rev.sent.value_counts().reindex(LABELS).replace(0, np.nan)
    rate = rt.groupby(["category", "sent"]).review_id.nunique().unstack(fill_value=0).reindex(columns=LABELS, fill_value=0).div(cls, axis=1) * 100
    overall = rt.groupby("category").review_id.nunique() / len(rev) * 100
    fig = go.Figure()
    for l in LABELS: fig.add_bar(x=rate.index, y=rate[l], name=l, marker_color=COLORS[l])
    fig.update_layout(barmode="group", yaxis_title="% of reviews mentioning topic"); display(_layout(fig, "Topic mention rate by review sentiment", 340))
    lift = pd.DataFrame({"mention rate %": overall, "negative lift": rate.negative / overall, "positive lift": rate.positive / overall}).round(2)
    display(lift)
    big = lift[overall >= 2]          # ignore rare topics (<2% of reviews)
    if len(big) and max((big["negative lift"] - 1).abs().max(), (big["positive lift"] - 1).abs().max()) < 0.3:
        display(HTML("<div style='padding:8px 12px;background:#eef4ff;border-left:4px solid #4C78A8;color:#111'><b>Data note:</b> every topic is mentioned at similar "
                     "rates in positive and negative reviews (lift within ±0.3), so the topic mix barely explains sentiment in this selection. "
                     "Treat topic 'drivers' with caution.</div>"))

def view_hotels(rev, sen, min_n=20):
    h = rev.groupby("hotel_name").agg(city=("location", "first"), stars=("star_rating", "first"), reviews=("review_id", "size"), avg_rating=("rating_score", "mean")).reset_index()
    h = h.merge(summarise(rev, "sent", "hotel_name")[["hotel_name", "negative", "neutral", "positive", "net"]], on="hotel_name").sort_values("net", ascending=False).round(2)
    display(h.rename(columns={"negative": "% neg", "neutral": "% neu", "positive": "% pos", "net": "net sentiment"}))
    display(net_bar(summarise(rev, "sent", "location"), "location", "Net sentiment by city")); display(net_bar(summarise(rev, "sent", "country"), "country", "Net sentiment by guest country"))
    display(heat(net_matrix(sen, "hotel_name", min_n), f"Net sentiment per hotel × topic (cells with < {min_n} mentions blank)", max(420, 24 * rev.hotel_name.nunique())))

def view_segments(rev, sen, dim="customer_type", min_n=20):
    ss = summarise(rev, "sent", dim); ss[dim] = ss[dim].astype(str)
    display(stacked(ss.sort_values("net"), dim, f"Sentiment mix by {dim}"))
    ar = rev.groupby(dim, observed=True).rating_score.mean().reset_index(); ar[dim] = ar[dim].astype(str)
    display(_layout(go.Figure(go.Bar(y=ar[dim], x=ar.rating_score, orientation="h", marker_color="#4C78A8")), f"Average rating by {dim}", 300))
    s2 = sen.copy(); s2[dim] = s2[dim].astype(str)
    display(heat(net_matrix(s2, dim, min_n), f"Topic net sentiment across {dim}", max(300, 50 * rev[dim].nunique())))

def view_models():
    bp = OUT / "baseline_best_per_config.csv"
    if bp.exists():
        b = pd.read_csv(bp); b["setup"] = np.where(b.use_rating, "text + rating", "text only")
        display(HTML("<b>Baseline sentiment classifiers</b> (test set; hyper-parameters chosen on validation)"))
        display(b[["model", "features", "setup", "hyperparam", "val_f1", "test_acc", "test_f1"]].head(10).round(4))
    for f in sorted(OUT.glob("predictions_bert_*.csv")):
        p = pd.read_csv(f); print("BERT", f.stem, "accuracy", round((p.true == p.pred).mean(), 4))
    ln = OUT / "label_noise_summary.json"
    if ln.exists():
        s = json.loads(ln.read_text()); a = s["accuracy"]
        display(HTML(f"<b>Label quality</b>: {s['pct_flagged']}% of labels ({s['n_flagged']:,}) flagged as likely wrong. "
                     f"Accuracy on raw test labels <b>{100 * a['B_train_clean__test_raw']:.1f}%</b>; with flagged rows removed from the test set "
                     f"<b>{100 * a['D_train_clean__test_clean_subset']:.1f}%</b> ({a['test_rows_flagged']}/{a['test_rows_total']} rows removed)."))
        display(HTML("<div style='padding:8px 12px;background:#fff6e5;border-left:4px solid #F58518;color:#111'>The higher figure is <b>optimistic</b> (rows removed by a similar model). "
                     "Report it only next to the raw-label accuracy and verify a sample of flagged rows by hand.</div>"))
        pd.options.display.max_colwidth = 160
        li = OUT / "label_issues.csv"
        if li.exists(): display(pd.read_csv(li).head(8)[["review_text", "rating_score", "sentiment_label", "suggested_label", "suggested_conf"]].round(2))

def view_explorer(rev, sen, q="", sentiments=(), topic="", show="All reviews", n=25):
    ex = rev
    if q: ex = ex[ex.review_text.str.contains(q, case=False, regex=False)]
    if sentiments: ex = ex[ex.sent.isin(sentiments)]
    if topic: ex = ex[ex.topics.str.contains(topic, regex=False)]
    if show == "Model ≠ given label": ex = ex[ex.sentiment_label != ex.pred_label]
    if show == "Suspected label issues": ex = ex[ex.label_issue]
    print(f"{len(ex):,} reviews match (showing {min(n, len(ex))})")
    pd.options.display.max_colwidth = 200
    display(ex[["review_id", "hotel_name", "review_date", "rating_score", "sentiment_label", "pred_label", "topics", "review_text"]].head(n))

# Part A · Static snapshot (no filters)

In [4]:
F0 = {"source": "Model-predicted", "d0": rev_all.review_date.min().date(), "d1": rev_all.review_date.max().date()}
rev0, sen0 = apply_filters(F0)
print("Sentiment source:", F0["source"])

Sentiment source: Model-predicted


### Overview

In [5]:
view_overview(rev0, sen0)

Model agrees with the given label on 94.2% of the reviews in view.


### Topics & topic-level sentiment

In [6]:
view_topics(rev0, sen0)

sent_pred,category,reviews,mentions,% neg,% neu,% pos,net,note
5,Pricing,13596,15292,14.4,69.4,16.2,1.8,
2,Facilities,9421,9665,24.4,48.7,26.9,2.5,
0,Cleanliness,6596,6725,29.1,38.2,32.7,3.6,
7,Service,11529,12145,24.4,47.2,28.5,4.1,
6,Room quality,6979,7017,21.8,52.0,26.3,4.5,
3,Food,6897,6931,21.3,49.4,29.3,7.9,
4,Location,9508,9813,23.1,41.6,35.4,12.3,
8,Staff,9162,9324,21.1,44.3,34.6,13.5,
1,Comfort,353,353,0.0,0.0,100.0,100.0,


,mention rate %,negative lift,positive lift
category,,,
Cleanliness,19.14,1.16,1.09
Comfort,1.02,0.00,3.03
Facilities,27.34,1.04,0.96
Food,20.01,0.92,1.02
Location,27.59,0.96,1.17
Pricing,39.45,0.79,0.74
Room quality,20.25,0.94,0.94
Service,33.46,1.02,0.99
Staff,26.59,0.90,1.16


### Hotels & locations

In [7]:
view_hotels(rev0, sen0)

,hotel_name,city,stars,reviews,avg_rating,% neg,% neu,% pos,net sentiment
11,Nottingham Riverside Hotel,Nottingham,5,1423,3.39,16.87,33.73,49.40,32.54
22,The London Heritage,London,5,1369,3.33,18.19,36.08,45.73,27.54
7,Liverpool Metropolitan,Liverpool,5,1274,3.31,18.13,36.97,44.90,26.77
8,London Metropolitan,London,5,1313,3.31,18.89,36.79,44.33,25.44
17,The Birmingham Heritage,Birmingham,4,1384,3.11,24.71,37.50,37.79,13.08
13,The Aberdeen Grand,Aberdeen,4,1336,3.13,23.73,39.82,36.45,12.72
16,The Birmingham Grand,Birmingham,4,1370,3.10,24.31,40.15,35.55,11.24
10,Nottingham Park Hotel,Nottingham,4,1410,3.11,23.26,42.62,34.11,10.85
3,Glasgow Metropolitan,Glasgow,4,1407,3.09,23.74,41.86,34.40,10.66
21,The London Boutique,London,4,1394,3.08,24.18,41.25,34.58,10.40


### Guest segments (example: loyalty status)

In [8]:
view_segments(rev0, sen0, "loyalty_status")

### Model performance & label quality

In [9]:
view_models()

,model,features,setup,hyperparam,val_f1,test_acc,test_f1
0,logreg,light_w12,text + rating,1.00,0.9468,0.9427,0.9441
1,logreg,light_w12_char,text + rating,1.00,0.9463,0.9427,0.9441
2,logreg,light_w13,text + rating,1.00,0.9462,0.9429,0.9442
3,linsvm,nostop_w12,text + rating,0.10,0.9460,0.9439,0.9452
4,logreg,nostop_w12,text + rating,1.00,0.9459,0.9454,0.9466
5,linsvm,light_w12_char,text + rating,0.10,0.9457,0.9427,0.9440
6,nb,nostop_w12,text + rating,0.03,0.9455,0.9441,0.9455
7,linsvm,light_w12,text + rating,0.10,0.9454,0.9425,0.9438
8,linsvm,light_w13,text + rating,0.10,0.9453,0.9433,0.9446
9,nb,light_w13,text + rating,0.03,0.9440,0.9423,0.9437


,review_text,rating_score,sentiment_label,suggested_label,suggested_conf
0,"Our time in Nottingham got off to a rocky start here. On the other hand, our arrival experience was delayed for no clear reason, which really affected our s...",1.4,neutral,negative,1.0
1,"We were disappointed that the location was disappointingly out of the way. On top of that, the rate we paid left a lot to be desired; we found better option...",2.2,neutral,negative,1.0
2,"Our time in Leicester got off to a rocky start here. On top of that, sadly, the breakfast turned out to be overpriced for the quality.",2.1,neutral,negative,1.0
3,"This hotel made our time in London truly memorable. On the other hand, i was genuinely impressed that the pricing was excellent value. Plus, honestly, the f...",4.4,neutral,positive,1.0
4,"This wasn't the experience we expected during our visit to Glasgow. On top of that, we were disappointed that the overall price was overpriced. Also, the ch...",2.2,neutral,negative,1.0
5,"Sadly, the breakfast turned out to be underwhelming. The overall price was a poor deal, which really affected our stay. Meanwhile, unfortunately, the gym wa...",1.7,neutral,negative,1.0
6,"The pricing left a lot to be desired; we found better options nearby for less. Also, the service team was dismissive, which really affected our stay. Left f...",1.7,neutral,negative,1.0
7,"The overall price was disappointing for the price, which really affected our stay. Sadly, the restaurant turned out to be overpriced for the quality. We'll ...",1.8,neutral,negative,1.0


### Review explorer (example: reviews where model ≠ given label)

In [10]:
view_explorer(rev0, sen0, show="Model ≠ given label", n=8)

2,010 reviews match (showing 8)


,review_id,hotel_name,review_date,rating_score,sentiment_label,pred_label,topics,review_text
10,11,Edinburgh Riverside Hotel,2025-03-01,3.7,positive,neutral,Pricing; Service; Staff,"The manager was reasonably polite for what we paid. Plus, arrival was fine for what we paid. Would definitely stay here again."
16,17,The Leeds Grand,2025-11-06,2.3,negative,neutral,Pricing; Service,"Overall our arrival experience was about average, about what we expected. Plus, the cost of the stay was reasonable enough for what we paid. We'll be looking elsewhere next time."
34,34,The London Heritage,2025-05-06,2.7,positive,neutral,Facilities,"This hotel was a fine, if unremarkable, choice for our time in London. Meanwhile, overall the Wi-Fi was nothing special, about what we expected."
40,40,The Belfast Boutique,2025-04-25,2.8,negative,neutral,Pricing,"Nothing about our visit to Belfast stood out much either way. Also, i'd describe the rate we paid as in line with similar hotels — nothing felt like a bargain or a rip-off."
68,67,Liverpool Harbour Hotel,2026-03-11,2.5,positive,neutral,Pricing; Service,"The cost of the stay was fair, nothing more and nothing less. On the other hand, i'd describe getting checked in as unremarkable — it took the usual amount of time."
76,75,The London Boutique,2025-10-18,2.6,negative,neutral,Pricing; Room quality; Service,"I'd describe the overall price as in line with similar hotels — it matched what similar hotels charge. Meanwhile, overall the room was adequately maintained, about what we expected. On the other h..."
108,106,Glasgow Harbour Hotel,2026-03-01,2.9,positive,neutral,Food,"I'd describe the breakfast as acceptable — it covered the basics fine. Might consider it again, might not."
111,109,Edinburgh Riverside Hotel,2026-02-12,4.1,neutral,positive,Location,"We couldn't have asked for a better base while visiting Edinburgh. The location was perfectly placed for exploring, and we could walk to most attractions within minutes."


---
# Part B · Live dashboard
Choose filters on the left, press **Apply**, then use the tabs. (Widgets need a live kernel — re-run the cell if they show as blank.)

In [11]:
dmin, dmax = rev_all.review_date.min().date(), rev_all.review_date.max().date()
W = {"source": widgets.ToggleButtons(options=["Model-predicted", "Given label"], description="Sentiment", style={"description_width": "70px"}),
     "d0": widgets.DatePicker(description="From", value=dmin), "d1": widgets.DatePicker(description="To", value=dmax),
     "drop_issues": widgets.Checkbox(description="Exclude suspected mislabelled reviews", indent=False)}
for col, label in FILTERS:
    W[col] = widgets.SelectMultiple(options=sorted(rev_all[col].dropna().unique().tolist()), description=label, rows=5,
                                    layout=widgets.Layout(width="330px"), style={"description_width": "90px"})
dim_dd = widgets.Dropdown(options=SEG_DIMS, value="customer_type", description="Segment by")
basis_dd = widgets.ToggleButtons(options=["Sentence-level", "Review-level"], description="Topic basis")
q_txt = widgets.Text(description="Search", placeholder="text in review…")
show_dd = widgets.Dropdown(options=["All reviews", "Model ≠ given label", "Suspected label issues"], description="Show")
topic_dd = widgets.Dropdown(options=[""] + sorted(sen_all.category.unique()), description="Topic")
apply_btn = widgets.Button(description="Apply filters", button_style="primary", icon="filter")
status = widgets.HTML()
TAB_NAMES = ["Overview", "Topics", "Hotels & locations", "Segments", "Models", "Explorer"]
outs = [widgets.Output() for _ in TAB_NAMES]
tabs = widgets.Tab(children=outs); tabs.titles = tuple(TAB_NAMES)
state = {}

def current_filters():
    f = {"source": W["source"].value, "d0": W["d0"].value or dmin, "d1": W["d1"].value or dmax, "drop_issues": W["drop_issues"].value}
    for col, _ in FILTERS: f[col] = list(W[col].value)
    return f

def render(i):
    rev, sen = state["rev"], state["sen"]
    with outs[i]:
        outs[i].clear_output(wait=True)
        if rev.empty: print("No reviews match the current filters."); return
        if i == 0: view_overview(rev, sen)
        elif i == 1: display(basis_dd); view_topics(rev, sen, basis_dd.value)
        elif i == 2: view_hotels(rev, sen)
        elif i == 3: display(dim_dd); view_segments(rev, sen, dim_dd.value)
        elif i == 4: view_models()
        elif i == 5: display(widgets.HBox([q_txt, topic_dd, show_dd])); view_explorer(rev, sen, q_txt.value, (), topic_dd.value, show_dd.value)

def refresh(_=None):
    status.value = "<i>updating…</i>"
    state["rev"], state["sen"] = apply_filters(current_filters())
    for i in range(len(outs)): render(i)
    status.value = f"<b>{len(state['rev']):,}</b> reviews · <b>{len(state['sen']):,}</b> topic mentions in view"

apply_btn.on_click(refresh)
dim_dd.observe(lambda c: render(3) if "rev" in state else None, names="value")
basis_dd.observe(lambda c: render(1) if "rev" in state else None, names="value")
for w in (q_txt, topic_dd, show_dd): w.observe(lambda c: render(5) if "rev" in state else None, names="value")

sidebar = widgets.VBox([widgets.HTML("<h4>Filters</h4>"), W["source"], widgets.HBox([W["d0"], W["d1"]]), *[W[c] for c, _ in FILTERS],
                        W["drop_issues"], apply_btn, status], layout=widgets.Layout(width="360px"))
display(widgets.HBox([sidebar, tabs], layout=widgets.Layout(align_items="flex-start")))
refresh()